# 04 – CẢI TIẾN TRÍCH XUẤT THÔNG TIN HÓA ĐƠN SROIE

Notebook này là sản phẩm chính của phần 04. Luật cải tiến nằm trực tiếp trong Notebook (mục 6); Baseline được import từ `src/baseline.py` của nhóm để bảo đảm đúng một nguồn luật.

**Quy tắc:** chỉ dùng Train và Validation, không nạp Test, không sửa Ground Truth. **Chọn luật trên Train, Validation chỉ để xác nhận** (không tinh chỉnh sau khi xem Validation). Mỗi cải tiến là một cờ riêng, được đo riêng (ablation); cờ nào không có lợi trên Train thì bị loại. Chỉ kết luận "tốt hơn" khi số liệu chứng minh.

## 1. Cấu hình và hàm đo

In [ ]:
from pathlib import Path
import re, json, itertools
from datetime import datetime
from collections import Counter
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "outputs").exists():
    for c in [PROJECT_DIR.parent, PROJECT_DIR / "project"]:
        if (c / "outputs").exists(): PROJECT_DIR = c; break

OCR_PATH = PROJECT_DIR / "outputs/ocr/train_validation_ocr.csv"
GT_TRAIN = PROJECT_DIR / "outputs/ground_truth/train_ground_truth.csv"
GT_VAL = PROJECT_DIR / "outputs/ground_truth/validation_ground_truth.csv"
BASE_TRAIN = PROJECT_DIR / "outputs/baseline/baseline_train.csv"
BASE_VAL = PROJECT_DIR / "outputs/baseline/baseline_validation.csv"
OUT_DIR = PROJECT_DIR / "outputs/improved"; OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [OCR_PATH, GT_TRAIN, GT_VAL]:
    assert p.exists(), f"Không tìm thấy file đầu vào: {p}"
BASE_OK = BASE_TRAIN.exists() and BASE_VAL.exists()
if not BASE_OK:
    print("CẢNH BÁO: thiếu baseline_train.csv/baseline_validation.csv của người 3 -> sẽ tự tính baseline từ src/baseline.py, KHÔNG đối chiếu được với file của người 3.")

import sys; sys.path.insert(0, str(PROJECT_DIR))
from src.baseline import extract_fields as extract_baseline, AMOUNT_RE, DATE_RE   # luật Baseline của người 3

FIELDS = ["company", "date", "address", "total"]

# ----- chuẩn hóa để đánh giá: GIỮ NGUYÊN như 03_baseline để số liệu so sánh được -----
DATE_FORMATS = ["%d/%m/%Y","%d-%m-%Y","%d.%m.%Y","%d/%m/%y","%d-%m-%y","%d.%m.%y","%Y-%m-%d","%Y/%m/%d",
                "%d %b %Y","%d%b%Y","%d-%b-%Y","%d %b %y","%d %B %Y"]
def is_missing(x):
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == ""
def norm_text(s):
    if is_missing(s): return None
    s = re.sub(r"[^\w\s]", " ", str(s).lower()); return re.sub(r"\s+", " ", s).strip()
def norm_total(s):
    if is_missing(s): return None
    nums = re.findall(r"\d[\d,.]*", str(s).replace(" ", ""))
    if not nums: return None
    tok = nums[-1]
    if re.search(r",\d{2}$", tok): tok = tok[:-3] + "." + tok[-2:]
    tok = tok.replace(",", "")
    try: return f"{float(tok):.2f}"
    except ValueError: return None
def norm_date(s):
    if is_missing(s): return None
    core = re.sub(r"\s*([/\-.])\s*", r"\1", str(s).strip())
    for cand in (core, core.replace(" ", "")):
        for f in DATE_FORMATS:
            try: return datetime.strptime(cand, f).strftime("%Y-%m-%d")
            except ValueError: pass
    return norm_text(core)
NORM = {"company": norm_text, "address": norm_text, "date": norm_date, "total": norm_total}
def exact(pred, gold, f):
    if is_missing(pred) or is_missing(gold): return 0
    return int(NORM[f](pred) == NORM[f](gold))

def token_f1(pred, gold):
    # F1 theo token (như 03_baseline) - đề yêu cầu báo F1 trường khi có quy tắc rõ
    p = (norm_text(pred) or "").split(); g = (norm_text(gold) or "").split()
    if not p or not g: return 0.0
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    pr, rc = common / len(p), common / len(g); return 2 * pr * rc / (pr + rc)

def gold_in_ocr(f, gold, text):
    """Thông tin đúng có nằm trong chữ OCR không? (cùng định nghĩa với 03_baseline)"""
    if is_missing(gold): return False
    if f == "total": return norm_total(gold) in {norm_total(a) for a in AMOUNT_RE.findall(text)}
    if f == "date": return norm_date(gold) in {norm_date(m.group(0)) for m in DATE_RE.finditer(text)}
    g = set((norm_text(gold) or "").split()); t = set((norm_text(text) or "").split())
    return len(g) > 0 and len(g & t) / len(g) >= 0.8
print("Thư mục dự án:", PROJECT_DIR)

## 2. Nạp và kiểm tra dữ liệu (chỉ Train/Validation)

In [ ]:
ocr = pd.read_csv(OCR_PATH, dtype={"key": str}).fillna({"ocr_text": ""})
gt = pd.concat([pd.read_csv(GT_TRAIN, dtype=str).fillna(""), pd.read_csv(GT_VAL, dtype=str).fillna("")], ignore_index=True)
assert ocr.key.is_unique and gt.key.is_unique, "Phát hiện key trùng"
assert set(ocr.split.unique()) <= {"train", "validation"}, "OCR có split ngoài Train/Validation"
assert set(ocr.key) == set(gt.key), "Key OCR và Ground Truth không khớp"
df = ocr.merge(gt[["key"] + FIELDS], on="key", how="inner", validate="one_to_one")
tr, va = df[df.split == "train"].reset_index(drop=True), df[df.split == "validation"].reset_index(drop=True)
print(f"Train={len(tr)}  Validation={len(va)}")

## 3. Tái lập baseline và đối chiếu với kết quả của người 3
Chạy lại `src/baseline.py` và **kiểm tra khớp từng ảnh** với `baseline_train.csv` / `baseline_validation.csv`. Nếu không khớp thì dừng, vì mọi so sánh phía sau sẽ vô nghĩa.

In [ ]:
def run_baseline(d):
    rows = []
    for _, r in d.iterrows():
        p = extract_baseline(r.ocr_text); row = {"key": r.key, "split": r.split}
        for f in FIELDS:
            row[f + "_pred"] = p.get(f); row[f + "_gold"] = r[f]; row[f + "_em"] = exact(p.get(f), r[f], f)
        rows.append(row)
    return pd.DataFrame(rows)
baseline = run_baseline(df)

if BASE_OK:
    ref = pd.concat([pd.read_csv(BASE_TRAIN, dtype={"key": str}), pd.read_csv(BASE_VAL, dtype={"key": str})]).set_index("key")
    chk = baseline.set_index("key")
    for f in FIELDS:
        diff = int((chk[f + "_em"] != ref.loc[chk.index, f + "_em"]).sum())
        assert diff == 0, f"Baseline tái lập KHÔNG khớp file của người 3 ở trường {f}: {diff} ảnh lệch"
    print("Baseline tái lập khớp 100% với baseline_train/validation.csv của người 3.")
else:
    print("Bỏ qua đối chiếu với file người 3 (thiếu file). Baseline dưới đây tính từ src/baseline.py.")
tbl = pd.DataFrame({s: {**{f: baseline[baseline.split == s][f + "_em"].mean() for f in FIELDS}} for s in ["train", "validation"]})
tbl.loc["MACRO"] = tbl.mean(); tbl.round(4)

## 4. Phân loại lỗi baseline: do OCR hay do luật?
`rule_error` = chữ đúng **có** trong OCR nhưng luật chọn sai (cải tiến luật sửa được). `ocr_error` = chữ đúng không có trong OCR (luật không sửa được; chỉ tiền xử lý ảnh/OCR khác mới giúp).

In [ ]:
def attribution(frame, texts, pred_em_col="_em", gold_col="_gold"):
    out = {}
    for f in FIELDS:
        c = []
        for (_, r), t in zip(frame.iterrows(), texts):
            if r[f + pred_em_col]: c.append("correct")
            elif gold_in_ocr(f, r[f + gold_col], t): c.append("rule_error")
            else: c.append("ocr_error")
        out[f] = pd.Series(c).value_counts()
    return pd.DataFrame(out).fillna(0).astype(int)
bv = baseline[baseline.split == "validation"].reset_index(drop=True)
bt = baseline[baseline.split == "train"].reset_index(drop=True)
print("Baseline – Train"); display(attribution(bt, tr.ocr_text))
print("Baseline – Validation"); display(attribution(bv, va.ocr_text))

In [ ]:
# Xem ví dụ lỗi do luật (Train) để thiết kế cải tiến có cơ sở
def show_rule_errors(field, n=3):
    shown = 0
    for (_, r), t in zip(bt.iterrows(), tr.ocr_text):
        if not r[field + "_em"] and gold_in_ocr(field, r[field + "_gold"], t):
            print("=" * 60); print("KEY:", r.key, "| ", field); print("DỰ ĐOÁN:", r[field + "_pred"]); print("ĐÁP ÁN :", r[field + "_gold"])
            print("--- OCR (800 ký tự đầu) ---"); print(t[:800]); shown += 1
            if shown >= n: break
show_rule_errors("total", 2)

## 5. Tiền xử lý ảnh (cần ảnh trong file parquet)
Theo `02_ocr.ipynb`, OCR CSV của người 1 được tạo bằng **Tesseract 5.5 (`--oem 3 --psm 6`, `eng`)** trên ảnh **grayscale + phóng 2x** (đó là mốc đối chứng ở đây). Người 1 thử nhanh trên **một** ảnh mẫu và thấy nhị phân Otsu **kém hơn nhiều** (WER 0.69 so với 0.35; độ tin cậy 51 so với 70), nên mục này đo lại trên nhiều ảnh các biến thể: `resize3x`, `deskew`, `clahe`, `median`, `otsu`, `adaptive`.

Cách đo: OCR lại một tập con ngẫu nhiên (`MAX_IMAGES` ảnh, `seed=42`) rồi so **tỉ lệ chữ đúng có trong OCR** (`gold_in_ocr`) với OCR gốc của người 1 trên đúng các ảnh đó. Ảnh đọc từ file Train parquet (file test **không** được đọc). Tốc độ khoảng 3 giây/ảnh/biến thể (người 1 mất ~32 phút cho 626 ảnh).
Quy trình: (1) đặt `RUN_PREPROC = True`, chạy so sánh trên Train; (2) nếu có biến thể thắng rõ, điền tên vào `FULL_OCR_VARIANT` để OCR lại toàn bộ 626 ảnh; (3) notebook chỉ chuyển sang OCR mới nếu Validation cũng tăng.

In [ ]:
# ---- tien xu ly anh + OCR (giong 02_ocr.ipynb cua nguoi 1: Tesseract, --oem 3 --psm 6, lang eng) ----
OCR_CONFIG = "--oem 3 --psm 6"
REF_STEPS = ["gray", "resize2"]      # dung nhu OCR CSV cua nguoi 1: grayscale + phong 2x (INTER_CUBIC)


def _deskew(gray):
    import cv2
    import numpy as np
    ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    ys, xs = np.where(ink > 0)
    if len(xs) < 200:
        return gray
    ang = cv2.minAreaRect(np.column_stack([xs, ys]).astype(np.float32))[-1]
    ang = ang - 90 if ang > 45 else ang
    if not 0.3 <= abs(ang) <= 15:
        return gray
    h, w = gray.shape[:2]

    def rot(a):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    def sharp(img):   # dong chu thang -> tong theo hang dao dong manh (tranh nham dau goc giua cac ban OpenCV)
        b = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        return float(np.var(b.sum(axis=1)))

    r1, r2 = rot(ang), rot(-ang)
    return max([(sharp(gray), gray), (sharp(r1), r1), (sharp(r2), r2)], key=lambda c: c[0])[1]


def preprocess_image(rgb, steps):
    """rgb: ndarray HxWx3 (RGB); steps: danh sach buoc ap dung theo thu tu."""
    import cv2
    out = rgb
    for st in steps:
        if st == "gray":
            out = cv2.cvtColor(out, cv2.COLOR_RGB2GRAY) if out.ndim == 3 else out
        elif st in ("resize2", "resize3"):
            f = 2 if st == "resize2" else 3
            out = cv2.resize(out, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
        elif st == "median":
            out = cv2.medianBlur(out, 3)
        elif st == "clahe":
            out = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(out)
        elif st == "otsu":
            out = cv2.threshold(cv2.GaussianBlur(out, (3, 3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
        elif st == "adaptive":
            out = cv2.adaptiveThreshold(cv2.GaussianBlur(out, (3, 3), 0), 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 31, 15)
        elif st == "deskew":
            out = _deskew(out)
        else:
            raise ValueError("buoc tien xu ly khong biet: " + st)
    return out


def ocr_image(pil_image, steps=None):
    import numpy as np
    import pytesseract
    steps = REF_STEPS if steps is None else steps
    img = preprocess_image(np.array(pil_image.convert("RGB")), steps)
    return pytesseract.image_to_string(img, lang="eng", config=OCR_CONFIG)

In [ ]:
RUN_PREPROC = False
PARQUET_TRAIN = PROJECT_DIR / "data" / "ICDAR2019-SROIE" / "data" / "train-00000-of-00001.parquet"   # như 02_ocr; KHÔNG đọc file test
PREPROC_EVAL_SPLIT = "train"      # chọn biến thể trên Train; đổi "validation" để xác nhận
MAX_IMAGES = 60                   # ~3 phút/biến thể; None = cả split
VARIANTS = {"resize3x": ["gray", "resize3"], "deskew": ["gray", "deskew", "resize2"], "clahe": ["gray", "resize2", "clahe"],
            "median": ["gray", "resize2", "median"], "otsu": ["gray", "resize2", "otsu"], "adaptive": ["gray", "resize2", "adaptive"]}
OCR_ORIG = dict(zip(df.key, df.ocr_text))     # OCR gốc của người 1 (mốc đối chứng, không bị ghi đè)

def _iter_images(keys):
    # đọc từng lô từ parquet Train, chỉ lấy các key cần (không nạp hết 626 ảnh vào RAM)
    import pyarrow.parquet as pq
    from PIL import Image
    from io import BytesIO
    keys = set(keys)
    for batch in pq.ParquetFile(PARQUET_TRAIN).iter_batches(batch_size=16, columns=["key", "image"]):
        for r in batch.to_pylist():
            if str(r["key"]) in keys:
                yield str(r["key"]), Image.open(BytesIO(r["image"]["bytes"])).convert("RGB")

def recall_table(frame, texts):
    return {f: float(np.mean([gold_in_ocr(f, g, texts[k]) for k, g in zip(frame.key, frame[f])])) for f in FIELDS}

if RUN_PREPROC:
    import time
    assert PARQUET_TRAIN.exists(), f"Không thấy {PARQUET_TRAIN}: sửa PARQUET_TRAIN cho đúng thư mục data của dự án"
    d = tr if PREPROC_EVAL_SPLIT == "train" else va
    sub = d.sample(n=min(MAX_IMAGES or len(d), len(d)), random_state=42).reset_index(drop=True)
    imgs = dict(_iter_images(sub.key))
    assert set(imgs) == set(sub.key), "Thiếu ảnh trong parquet"
    # kiểm tra tái tạo: OCR lại bằng đúng cấu hình của người 1 có ra đúng văn bản trong CSV không?
    same = sum(ocr_image(imgs[k], REF_STEPS).strip() == OCR_ORIG[k].strip() for k in list(imgs)[:3])
    print(f"Tái tạo OCR gốc: {same}/3 ảnh giống hệt CSV của người 1 (nếu 0/3, phiên bản Tesseract/OpenCV khác -> so sánh dưới đây vẫn dùng văn bản CSV làm mốc nhưng chênh lệch có thể do phiên bản).")
    rec = {"control: gray+2x (CSV người 1)": recall_table(sub, OCR_ORIG)}
    rec["control: gray+2x (CSV người 1)"]["giay"] = 0
    for name, steps in VARIANTS.items():
        t0 = time.time(); txt = {k: ocr_image(img, steps) for k, img in imgs.items()}
        rec[name] = {**recall_table(sub, txt), "giay": round(time.time() - t0)}
        pd.DataFrame({"key": list(txt), "ocr_text": list(txt.values())}).to_csv(OUT_DIR / f"ocr_subset_{name}.csv", index=False, encoding="utf-8-sig")
    pre = pd.DataFrame(rec).T; pre["mean"] = pre[FIELDS].mean(axis=1)
    pre["delta_vs_control"] = pre["mean"] - pre.loc["control: gray+2x (CSV người 1)", "mean"]
    print(f"Tỉ lệ chữ đúng có trong OCR, {PREPROC_EVAL_SPLIT}, n={len(sub)}:"); display(pre.round(4))
    pre.round(4).to_csv(OUT_DIR / "tien_xu_ly_anh_so_sanh.csv", encoding="utf-8-sig")
else:
    print("Bỏ qua tiền xử lý ảnh (RUN_PREPROC=False) – chưa có kết quả, KHÔNG được kết luận về grayscale/resize/threshold/deskew.")

In [ ]:
# OCR lại TOÀN BỘ 626 ảnh bằng biến thể thắng (~30 phút). Điền tên biến thể vào đây sau khi xem bảng ở trên.
FULL_OCR_VARIANT = None           # ví dụ "deskew"
FULL_PATH = OUT_DIR / f"ocr_full_{FULL_OCR_VARIANT}.csv" if FULL_OCR_VARIANT else None
if FULL_OCR_VARIANT:
    import time
    steps = VARIANTS[FULL_OCR_VARIANT]
    done = pd.read_csv(FULL_PATH, dtype={"key": str}).fillna("") if FULL_PATH.exists() else pd.DataFrame(columns=["key", "ocr_text"])
    todo = [k for k in df.key if k not in set(done.key)]; rows = done.to_dict("records"); t0 = time.time()   # chạy lại được nếu bị ngắt
    for i, (k, img) in enumerate(_iter_images(todo), 1):
        rows.append({"key": k, "ocr_text": ocr_image(img, steps)})
        if i % 25 == 0 or i == len(todo):
            pd.DataFrame(rows).to_csv(FULL_PATH, index=False, encoding="utf-8-sig"); print(f"{i}/{len(todo)}  {round(time.time() - t0)}s")
    pd.DataFrame(rows).to_csv(FULL_PATH, index=False, encoding="utf-8-sig")

# Chỉ chuyển sang OCR mới khi Validation cũng tăng (so với OCR gốc của người 1)
OCR_SOURCE, PREPROC_STEPS = "original (gray+resize2x)", REF_STEPS
if FULL_PATH is not None and FULL_PATH.exists():
    new = pd.read_csv(FULL_PATH, dtype={"key": str}).fillna("").set_index("key")["ocr_text"].to_dict()
    if set(df.key) <= set(new):
        r_old, r_new = recall_table(va, OCR_ORIG), recall_table(va, new)
        cmp_ocr = pd.DataFrame({"gốc": r_old, FULL_OCR_VARIANT: r_new}).T; cmp_ocr["mean"] = cmp_ocr[FIELDS].mean(axis=1); display(cmp_ocr.round(4))
        if cmp_ocr.loc[FULL_OCR_VARIANT, "mean"] > cmp_ocr.loc["gốc", "mean"] + 0.005:
            df["ocr_text"] = df.key.map(new); tr, va = df[df.split == "train"].reset_index(drop=True), df[df.split == "validation"].reset_index(drop=True)
            OCR_SOURCE, PREPROC_STEPS = FULL_OCR_VARIANT, VARIANTS[FULL_OCR_VARIANT]
            print("ĐÃ chuyển sang OCR mới:", OCR_SOURCE)
        else:
            print("OCR mới KHÔNG tốt hơn rõ rệt trên Validation -> giữ OCR gốc.")
    else:
        print("File OCR đầy đủ chưa phủ hết key -> giữ OCR gốc.")
print("Nguồn OCR dùng cho các mục sau:", OCR_SOURCE)

## 6. Luật cải tiến (mỗi cải tiến là một cờ)

**Nguyên tắc: khi mọi cờ tắt, kết quả GIỐNG HỆT baseline của người 3** (luật baseline được sao chép nguyên văn vào module và có `assert` kiểm tra từng ảnh). Nhờ vậy mọi mức tăng/giảm trong ablation là so với baseline thật.

| Cờ | Cải tiến | Lỗi nhắm tới (từ ghi chú của người 3) |
|---|---|---|
| `clean_text` | bỏ ký tự rác đầu dòng, `\x0c`, dòng chỉ có ký hiệu (áp cho mọi trường) | OCR nhiễu |
| `clean_date` | sửa nhầm O/0, l/1, S/5… trong chuỗi ngày | date: lỗi đọc chữ |
| `date_v2` | kiểm tra ngày hợp lệ (loại `4 - 6/300`), nhiều định dạng, ưu tiên dòng `DATE`, bỏ ngày hết hạn; không tìm được thì dùng luật baseline | date sai/None |
| `clean_amount` | sửa nhầm ký tự trong số tiền | total: `9.O0` |
| `total_kw` | thêm từ khóa (`NETT TOTAL`, `TOTAL INCL`, `TOTAL DUE`…) và không loại nhầm `TOTAL INCL. GST` | total chọn sai dòng/None |
| `total_nospace` | so khớp từ khóa sau khi bỏ khoảng trắng (`NetTotal`, `GRANDTOTAL`) | total None do từ khóa dính liền |
| `total_gstcut` | bỏ phần `GST/TAX SUMMARY` khi tìm total | lấy nhầm số trong bảng GST |
| `total_pos` | không có từ khóa → số lớn nhất ở phần dưới hóa đơn | total None |
| `company_v2` | chấm điểm dòng đầu (hậu tố SDN BHD…, chữ HOA, loại TEL/INVOICE/địa chỉ, phạt dòng toàn chữ thường như tên người), ghép `SDN BHD` bị tách dòng | company = dòng 0 là tên người |
| `addr_v2` | dấu hiệu địa chỉ rộng hơn, cắt phần `TEL/FAX` cuối dòng | địa chỉ thiếu/thừa |
| `addr_merge` | lấy **khối dòng liên tiếp** (cho phép hỏng 1 dòng) thay vì gom rời rạc | địa chỉ thừa dòng |

Quan trọng: address luôn được tính **từ vị trí dòng company đã chọn** (kể cả khi tắt mọi cờ address, vẫn dùng luật baseline nhưng bắt đầu từ dòng company mới), nên `company_v2` đúng thì address cũng hết lẫn tên công ty.

In [ ]:
import re
from datetime import date as _date

FIELDS = ["company", "date", "address", "total"]

DEFAULT_CFG = dict(
    clean_text=False,     # lam sach dong OCR: bo ky tu rac dau dong, \x0c, dong chi co ky hieu
    clean_date=False,     # sua nham O/0, l/1, S/5... trong chuoi ngay (truoc khi tim ngay)
    date_v2=False,        # kiem tra ngay hop le, nhieu dinh dang, uu tien dong DATE, bo ngay het han
    clean_amount=False,   # sua nham ky tu so trong so tien (truoc khi tim total)
    total_kw=False,       # tu khoa total mo rong + khong loai nham 'TOTAL INCL GST'
    total_nospace=False,  # so khop tu khoa sau khi bo khoang trang (NetTotal, GRANDTOTAL)
    total_gstcut=False,   # bo phan GST/TAX SUMMARY khi tim total
    total_pos=False,      # khong co tu khoa -> so lon nhat o phan duoi hoa don
    company_v2=False,     # cham diem cac dong dau de chon ten cong ty (+ ghep SDN BHD tach dong)
    addr_v2=False,        # dia chi: dau hieu dia chi rong hon, cat phan TEL/FAX cuoi dong
    addr_merge=False,     # dia chi: lay KHOI dong lien tiep (cho phep hong 1 dong) thay vi gom roi rac
)

# ================================================================ BASELINE (sao chep tu src/baseline.py cua nguoi 3)
MONTHS = r"(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*"
DATE_PATTERNS = [
    r"\b\d{1,2}\s*[/\-.]\s*\d{1,2}\s*[/\-.]\s*\d{2,4}\b",
    r"\b\d{4}\s*[/\-.]\s*\d{1,2}\s*[/\-.]\s*\d{1,2}\b",
    r"\b\d{1,2}[\s\-]*" + MONTHS + r"[\s\-,]*\d{2,4}\b",
]
DATE_RE = re.compile("|".join(f"(?:{p})" for p in DATE_PATTERNS), re.IGNORECASE)
AMOUNT_RE = re.compile(r"\d[\d,]*[.,]\s?\d{2}\b")
TOTAL_KEYS = ["GRAND TOTAL", "TOTAL ROUNDED", "ROUNDED TOTAL", "NET TOTAL",
              "TOTAL AMOUNT", "AMOUNT DUE", "AMOUNT PAYABLE", "TOTAL"]
TOTAL_EXCLUDE = re.compile(r"SUB|EXCL|QTY|QUANTITY|SAVING|DISCOUNT|POINT|TOTAL\s*GST|TAX", re.IGNORECASE)
ADDR_KW = re.compile(
    r"\b(JALAN|JLN|TAMAN|TMN|LOT|NO|BANDAR|KAWASAN|SELANGOR|JOHOR|KUALA LUMPUR|KL|"
    r"PETALING|SHAH ALAM|KLANG|CHERAS|PAHANG|PENANG|WILAYAH)\b", re.IGNORECASE)
POSTCODE = re.compile(r"\b\d{5}\b")
ADDR_STOP = re.compile(r"\b(TEL|FAX|GST|INVOICE|RECEIPT|DATE|CASH|BILL|TAX|DOC|CASHIER)\b", re.IGNORECASE)


def _get_lines(ocr_text):
    if not isinstance(ocr_text, str):
        return []
    return [ln.strip() for ln in ocr_text.split("\n") if ln.strip()]


def _b_company(lines):
    for i, ln in enumerate(lines[:5]):
        if sum(c.isalpha() for c in ln) >= 3:
            return ln, i
    return None, -1


def _b_date(lines):
    for ln in lines:
        if "DATE" in ln.upper():
            m = DATE_RE.search(ln)
            if m:
                return m.group(0).strip()
    m = DATE_RE.search("\n".join(lines))
    return m.group(0).strip() if m else None


def _b_address(lines, company_idx):
    if company_idx < 0:
        return None
    parts = []
    for ln in lines[company_idx + 1: company_idx + 9]:
        if ADDR_STOP.search(ln):
            break
        if ADDR_KW.search(ln) or POSTCODE.search(ln):
            parts.append(ln)
    return " ".join(parts) if parts else None


# ================================================================ lam sach
_CONF = str.maketrans("OoIl|SsBZ", "001115582")
_TOKEN = re.compile(r"[0-9A-Za-z|./,\-]{4,}")
_ALLOWED = set("0123456789OoIl|SsBZ./,-")


def _digit_fix(line):
    """Trong token co >= 2 chu so va chi gom ky tu de nham, doi O->0, l->1, S->5..."""
    def fn(m):
        t = m.group(0)
        if sum(c.isdigit() for c in t) >= 2 and all(c in _ALLOWED for c in t):
            return t.translate(_CONF)
        return t
    return _TOKEN.sub(fn, line)


def _clean_lines(lines):
    out = []
    for l in lines:
        l = l.replace("\x0c", " ")
        l = re.sub(r"^[^A-Za-z0-9(]+", "", l)
        l = re.sub(r"\s+", " ", l).strip()
        if sum(c.isalnum() for c in l) >= 2:
            out.append(l)
    return out


# ================================================================ date
_MONTHS = {m: i + 1 for i, m in enumerate("JAN FEB MAR APR MAY JUN JUL AUG SEP OCT NOV DEC".split())}
_MON = "|".join(_MONTHS)
_P_NUM = re.compile(r"(?<!\d)(\d{1,2})\s*[/\-.]\s*(\d{1,2})\s*[/\-.]\s*(\d{4}|\d{2})(?!\d)")
_P_ISO = re.compile(r"(?<!\d)(\d{4})\s*[/\-.]\s*(\d{1,2})\s*[/\-.]\s*(\d{1,2})(?!\d)")
_P_TXT = re.compile(r"(?<!\d)(\d{1,2})\s*[\- ]?\s*(" + _MON + r")[A-Z]*\.?[\s,\-]*(\d{4}|\d{2})(?!\d)")
_P_TXT2 = re.compile(r"(" + _MON + r")[A-Z]*\.?\s*(\d{1,2})(?:ST|ND|RD|TH)?[\s,\-]+(\d{4})(?!\d)")


def _mk(y, m, d):
    y, m, d = int(y), int(m), int(d)
    if y < 100:
        y += 2000
    if not 2000 <= y <= 2030:
        return None
    try:
        return _date(y, m, d).isoformat()
    except ValueError:
        return None


def _find_dates_line(u):
    """-> list[(vi_tri, iso, chuoi_goc)] chi gom ngay HOP LE."""
    out = []
    for m in _P_ISO.finditer(u):
        iso = _mk(*m.groups())
        if iso:
            out.append((m.start(), iso, m.group(0)))
    for m in _P_NUM.finditer(u):
        a, b, c = m.groups()
        a_, b_ = int(a), int(b)
        if b_ > 12 and a_ <= 12:
            a_, b_ = b_, a_
        iso = _mk(c, b_, a_)
        if iso:
            out.append((m.start(), iso, m.group(0)))
    for m in _P_TXT.finditer(u):
        d, mon, y = m.groups()
        iso = _mk(y, _MONTHS[mon], d)
        if iso:
            out.append((m.start(), iso, m.group(0)))
    for m in _P_TXT2.finditer(u):
        mon, d, y = m.groups()
        iso = _mk(y, _MONTHS[mon], d)
        if iso:
            out.append((m.start(), iso, m.group(0)))
    return sorted(out)


def _v2_date(lines):
    best = None
    for i, l in enumerate(lines):
        u = l.upper()
        for pos, iso, raw in _find_dates_line(u):
            score = 2 if "DATE" in u else 0
            if re.search(r"EXP|DUE|VALID|BEST BEFORE", u):
                score -= 2
            key = (score, -i, -pos)
            if best is None or key > best[0]:
                best = (key, raw)
    return best[1].strip() if best else None


def extract_date(lines, cfg):
    if cfg["clean_date"]:
        lines = [_digit_fix(l) for l in lines]
    if cfg["date_v2"]:
        d = _v2_date(lines)
        if d:
            return d
    return _b_date(lines)


# ================================================================ total
_TOTAL_KEYS_V2 = ["GRAND TOTAL", "TOTAL ROUNDED", "ROUNDED TOTAL", "NET TOTAL", "NETT TOTAL", "TOTAL AMOUNT",
                  "AMOUNT DUE", "TOTAL DUE", "AMOUNT PAYABLE", "TOTAL PAYABLE", "TOTAL INCL", "TOTAL SALES", "TOTAL"]
_TOTAL_EXCLUDE_V2 = re.compile(r"SUB|EXCL|QTY|QUANTITY|SAVING|DISCOUNT|POINT|CHANGE|TENDER|TOTAL\s*(?:GST|TAX)|"
                               r"(?:GST|TAX)\s*TOTAL", re.IGNORECASE)
_GST_HEAD = re.compile(r"(?:GST|TAX)\s*(?:SUMMARY|ANALYSIS)", re.IGNORECASE)


def _amount_value(a):
    s = a.replace(" ", "")
    if re.search(r",\d{2}$", s):
        s = s[:-3] + "." + s[-2:]
    return float(s.replace(",", ""))


def extract_total(lines, cfg):
    if cfg["clean_amount"]:
        lines = [_digit_fix(l) for l in lines]
    if cfg["total_gstcut"]:
        for i, l in enumerate(lines):
            if _GST_HEAD.search(l):
                lines = lines[:i]
                break
    keys = _TOTAL_KEYS_V2 if cfg["total_kw"] else TOTAL_KEYS
    excl = _TOTAL_EXCLUDE_V2 if cfg["total_kw"] else TOTAL_EXCLUDE
    ns = cfg["total_nospace"]
    for key in keys:
        k = key.replace(" ", "") if ns else key
        for i, ln in enumerate(lines):
            up = ln.upper().replace(" ", "") if ns else ln.upper()
            if k in up and not excl.search(ln.upper()):
                amounts = AMOUNT_RE.findall(ln)
                if not amounts and i + 1 < len(lines):
                    amounts = AMOUNT_RE.findall(lines[i + 1])
                if amounts:
                    return amounts[-1].replace(" ", "")
    if cfg["total_pos"]:
        n = len(lines)
        cands = []
        for i, ln in enumerate(lines):
            if i >= n * 0.4 and not excl.search(ln.upper()):
                for a in AMOUNT_RE.findall(ln):
                    try:
                        cands.append((_amount_value(a), a))
                    except ValueError:
                        pass
        if cands:
            return max(cands)[1].replace(" ", "")
    return None


# ================================================================ company
_LEGAL = re.compile(r"\b(SDN\.?\s*BHD|BHD|S/?B|ENTERPRISE|TRADING|RESTAURANT|RESTORAN|STORE|MARKET|MART|CAFE|"
                    r"BAKERY|PHARMACY|LTD|INC|CO\.?|PLT|CORPORATION|HOLDINGS|SUPERMARKET|HARDWARE|BOOK|"
                    r"GIFT|HOME|MACHINERY|SHOP|MEDICAL|CLINIC|KEDAI)\b")
_NOISE = re.compile(r"\b(TEL|FAX|PHONE|INVOICE|RECEIPT|TAX|DATE|GST|REG|CASHIER|DOC|BILL|WWW|EMAIL|HP)\b|\bNO\s*[:.]")
_ADDR_CUE = re.compile(r"\b(JALAN|JLN|LORONG|LRG|TAMAN|TMN|BANDAR|KAWASAN|LOT|FLOOR|LEVEL|BLOCK|BLK|MENARA|WISMA|"
                       r"SEKSYEN|PUSAT|KOMPLEKS|MALAYSIA|KUALA LUMPUR|SELANGOR|JOHOR|PENANG|PERAK|KEDAH|SABAH|"
                       r"SARAWAK|MELAKA|PAHANG|PETALING|SHAH ALAM|KLANG|CHERAS|WILAYAH|KL|NO)\b", re.IGNORECASE)
_ADDR_STOP2 = re.compile(r"\b(TEL|FAX|PHONE|GST|INVOICE|RECEIPT|DATE|DOC|CASHIER|EMAIL|WWW|HP|BILL|SLIP|CASH|TAX)\b",
                         re.IGNORECASE)


def _v2_company(lines):
    best = None
    for rank, i in enumerate(range(min(len(lines), 8))):
        l = lines[i].strip()
        u = l.upper()
        if len(l) < 3 or sum(c.isalpha() for c in l) < 3:
            continue
        s = -0.3 * rank
        if _LEGAL.search(u):
            s += 3
        if _NOISE.search(u):
            s -= 4
        if _ADDR_CUE.search(u) or re.search(r"\b\d{5}\b", u):
            s -= 3
        if sum(c.isdigit() for c in l) / len(l) > 0.3:
            s -= 2
        letters = [c for c in l if c.isalpha()]
        up = sum(c.isupper() for c in letters) / len(letters)
        s += 1 if up >= 0.8 else (-1.5 if up == 0 else 0)
        if best is None or s > best[0]:
            best = (s, i)
    if best is None:
        return None, -1
    i = best[1]
    name, end = lines[i].strip(), i
    if (i + 1 < len(lines) and not _LEGAL.search(name.upper())
            and re.match(r"^\s*(SDN\.?\s*BHD|BHD|\(?S/?B\)?)\b", lines[i + 1].upper())):
        name, end = name + " " + lines[i + 1].strip(), i + 1
    return name, end


# ================================================================ address
def _cut_tail(l):
    return re.split(r"\b(?:TEL|PHONE|FAX|HP)\b", l, maxsplit=1, flags=re.I)[0].strip(" ,:-")


def extract_address(lines, cidx, cfg):
    if cidx < 0:
        return None
    if not (cfg["addr_v2"] or cfg["addr_merge"]):
        return _b_address(lines, cidx)           # = baseline, nhung tinh tu dong company da chon
    v2 = cfg["addr_v2"]
    kw = _ADDR_CUE if v2 else ADDR_KW
    stop = _ADDR_STOP2 if v2 else ADDR_STOP
    window = lines[cidx + 1: cidx + 9]
    fix = _cut_tail if v2 else (lambda x: x)
    if not cfg["addr_merge"]:                    # chi addr_v2: gom nhu baseline nhung dau hieu rong hon + cat TEL
        parts = []
        for ln in window:
            if stop.search(ln):
                break
            if kw.search(ln) or POSTCODE.search(ln):
                parts.append(fix(ln))
        return " ".join(p for p in parts if p) or None
    # addr_merge: lay KHOI dong lien tiep co dau hieu dia chi, cho phep hong toi da 1 dong giua khoi
    parts, started, gap = [], False, 0
    for ln in window:
        if stop.search(ln):
            break
        if kw.search(ln) or POSTCODE.search(ln):
            parts.append(fix(ln))
            started, gap = True, 0
        elif started:
            gap += 1
            if gap > 1:
                break
            parts.append(("?", fix(ln)))
    out = []
    for p in parts:
        out.append(p)
    # bo dong 'hong' o cuoi khoi, giu dong 'hong' nam giua hai dong dia chi
    while out and isinstance(out[-1], tuple):
        out.pop()
    return " ".join(p[1] if isinstance(p, tuple) else p for p in out if (p[1] if isinstance(p, tuple) else p)) or None


# ================================================================ API
def extract(text, cfg=None):
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    lines = _get_lines(text)
    if cfg["clean_text"]:
        lines = _clean_lines(lines)
    if cfg["company_v2"]:
        company, cidx = _v2_company(lines)
    else:
        company, cidx = _b_company(lines)
    return dict(company=company, date=extract_date(lines, cfg),
                address=extract_address(lines, cidx, cfg), total=extract_total(lines, cfg))

In [ ]:
# Kiểm tra nhanh luật cải tiến
demo = "tan woon yann\nABC TRADING SDN BHD\nNO 1 JALAN BUNGA\n50000 KUALA LUMPUR\nTEL: 012\nDATE: 25/12/2018\nSUBTOTAL 8.00\nNetTotal 9.00\nGST SUMMARY\nSR 6% 0.50 TOTAL 9.50"
full = {k: True for k in DEFAULT_CFG}
print(extract(demo, full)); assert extract(demo, full)["total"] == "9.00" and extract(demo, full)["company"] == "ABC TRADING SDN BHD"
print("OK")

# BẮT BUỘC: mọi cờ tắt phải cho kết quả giống hệt baseline của người 3, từng ảnh
mism = [k for k, t in zip(df.key, df.ocr_text) if extract(t, DEFAULT_CFG) != extract_baseline(t)]
assert not mism, f"Luật mọi-cờ-tắt KHÁC baseline ở {len(mism)} ảnh, ví dụ: {mism[:3]}"
print("Mọi cờ tắt == baseline của người 3 trên", len(df), "ảnh.")

## 7. Ablation – từng cải tiến đo riêng
Mốc là **baseline** (mọi cờ tắt = đúng luật của người 3). Mỗi cờ được bật **một mình** rồi đo cả 4 trường; cột `Δ…` là chênh lệch so với baseline. Một số cờ chỉ có tác dụng khi đi cùng cờ khác (ví dụ `addr_merge` cần dòng company đúng), nên có thêm bảng **tích lũy** theo thứ tự. Cột `Train` dùng để chọn; `Validation` chỉ để xem/xác nhận.

In [ ]:
def predict(d, cfg):
    return pd.DataFrame([extract(t, cfg) for t in d["ocr_text"]], index=d.index)
def evaluate(d, cfg):
    p = predict(d, cfg)
    e = {f: float(np.mean([exact(a, b, f) for a, b in zip(p[f], d[f])])) for f in FIELDS}
    e["macro"] = float(np.mean([e[f] for f in FIELDS])); return e
COLS = FIELDS + ["macro"]
FLAG_FIELD = {"clean_text":"macro","clean_date":"date","date_v2":"date","clean_amount":"total","total_kw":"total",
              "total_nospace":"total","total_gstcut":"total","total_pos":"total","company_v2":"company",
              "addr_v2":"address","addr_merge":"address"}
BASE = "baseline (mọi cờ tắt)"
rows = []
def add(name, cfg):
    for sp, d in [("Train", tr), ("Validation", va)]:
        e = evaluate(d, cfg); rows.append(dict(config=name, split=sp, **{c: round(e[c], 4) for c in COLS}))
add(BASE, DEFAULT_CFG)
for flag in DEFAULT_CFG: add(f"+ {flag}", {**DEFAULT_CFG, flag: True})
add("tất cả cờ bật", {k: True for k in DEFAULT_CFG})
abl = pd.DataFrame(rows)
def show_abl(table):
    for sp in ["Train", "Validation"]:
        a = table[table.split == sp].set_index("config").drop(columns="split")
        d = (a[COLS] - a.loc[a.index[0], COLS]).round(4).add_prefix("Δ")
        print(f"--- {sp} ---"); display(pd.concat([a, d], axis=1))
print("Nguồn OCR:", OCR_SOURCE, "(nếu khác 'original' thì dòng baseline ở đây là LUẬT baseline trên OCR mới; mục 9 so với baseline gốc của người 3)")
print("ABLATION TỪNG CỜ (bật một mình):"); show_abl(abl)
abl.to_csv(OUT_DIR / "ablation_tung_co.csv", index=False, encoding="utf-8-sig")

# Tích lũy theo thứ tự (cờ phụ thuộc nhau mới lộ tác dụng)
ORDER = ["clean_text","clean_date","date_v2","clean_amount","total_kw","total_nospace","total_gstcut","total_pos",
         "company_v2","addr_v2","addr_merge"]
rows2, cur = [], dict(DEFAULT_CFG)
for sp, d in [("Train", tr), ("Validation", va)]:
    e = evaluate(d, cur); rows2.append(dict(config=BASE, split=sp, **{c: round(e[c], 4) for c in COLS}))
for flag in ORDER:
    cur[flag] = True
    for sp, d in [("Train", tr), ("Validation", va)]:
        e = evaluate(d, cur); rows2.append(dict(config=f"+ {flag} (tích lũy)", split=sp, **{c: round(e[c], 4) for c in COLS}))
abl_cum = pd.DataFrame(rows2)
print("ABLATION TÍCH LŨY:"); show_abl(abl_cum)
abl_cum.to_csv(OUT_DIR / "ablation_tich_luy.csv", index=False, encoding="utf-8-sig")

## 8. Chọn phương pháp trên **Train** và đóng băng cấu hình
Duyệt mọi tổ hợp cờ theo nhóm **trên Train**: `clean_text` (đo bằng macro 4 trường) → date → total → company → address (address dùng company đã chọn). Mốc so sánh là baseline (mọi cờ tắt). Chọn EM Train cao nhất; hòa thì chọn ít cờ hơn; chỉ bật khi hơn "tắt hết" ít nhất `MIN_GAIN` (0.5 điểm phần trăm). Validation **không** tham gia chọn.

In [ ]:
MIN_GAIN = 0.005
SELECT_ON = "train"   # "train" (khuyên dùng: Validation chỉ để xác nhận) hoặc "validation"
assert SELECT_ON in ("train", "validation")
SEL = tr if SELECT_ON == "train" else va
GROUPS = [("macro", ["clean_text"]), ("date", ["clean_date","date_v2"]), ("total", ["clean_amount","total_kw","total_nospace","total_gstcut","total_pos"]),
          ("company", ["company_v2"]), ("address", ["addr_v2","addr_merge"])]
chosen, log = dict(DEFAULT_CFG), []
for field, flags in GROUPS:
    results = []
    for combo in itertools.product([False, True], repeat=len(flags)):
        cfg = {**chosen, **dict(zip(flags, combo))}
        em = evaluate(SEL, cfg)[field]; results.append((em, -sum(combo), combo))
        log.append(dict(field=field, **dict(zip(flags, combo)), EM_chon=round(em, 4)))
    best = max(results); off = next(r for r in results if not any(r[2]))
    use = best[2] if best[0] >= off[0] + MIN_GAIN else off[2]
    chosen.update(dict(zip(flags, use)))
    print(f"{field:8s}: baseline={off[0]:.4f} -> chọn {dict(zip(flags, use))}  EM_chon={max(r for r in results if r[2]==use)[0]:.4f}")
pd.DataFrame(log).to_csv(OUT_DIR / "tim_kiem_to_hop_co_train.csv", index=False)
FROZEN = dict(cfg=chosen, selected_on=SELECT_ON, min_gain=MIN_GAIN, n_train=len(tr), n_validation=len(va), ocr_source=OCR_SOURCE,
              ocr=dict(steps=PREPROC_STEPS, tesseract_config=OCR_CONFIG, lang="eng"))
(OUT_DIR / "frozen_config.json").write_text(json.dumps(FROZEN, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(chosen, indent=1))

In [ ]:
# Từng cờ trong cấu hình cuối đóng góp bao nhiêu (tắt từng cờ), trên Train và Validation
fin_tr, fin_va = evaluate(tr, chosen), evaluate(va, chosen)
loo = []
for flag, on in chosen.items():
    if on:
        f = FLAG_FIELD[flag]; etr, eva = evaluate(tr, {**chosen, flag: False}), evaluate(va, {**chosen, flag: False})
        loo.append(dict(co=flag, truong=f, dong_gop_train=round(fin_tr[f] - etr[f], 4), dong_gop_validation=round(fin_va[f] - eva[f], 4)))
loo = pd.DataFrame(loo); display(loo) if len(loo) else print("Không có cờ nào được bật.")

## 9. So sánh baseline và cải tiến trên cùng Validation (đã đóng băng)

In [ ]:
pv, pt = predict(va, chosen), predict(tr, chosen)
def compare(d, p, b):
    rows = []
    for f in FIELDS:
        imp = np.array([exact(a, g, f) for a, g in zip(p[f], d[f])]); bas = b[f + "_em"].to_numpy()
        rows.append(dict(field=f, baseline=bas.mean(), cai_tien=imp.mean(), chenh_lech=imp.mean() - bas.mean(),
                         sua_dung=int(((bas == 0) & (imp == 1)).sum()), lam_hong=int(((bas == 1) & (imp == 0)).sum())))
    c = pd.DataFrame(rows).set_index("field")
    c.loc["MACRO"] = [c.baseline.mean(), c.cai_tien.mean(), c.chenh_lech.mean(), c.sua_dung.sum(), c.lam_hong.sum()]
    return c.round(4)
cmp_val, cmp_tr = compare(va, pv, bv), compare(tr, pt, bt)
print("VALIDATION (n=%d)" % len(va)); display(cmp_val)
print("TRAIN (n=%d)" % len(tr)); display(cmp_tr)
cmp_val.assign(split="validation").reset_index().to_csv(OUT_DIR / "validation_and_train_comparison.csv", index=False, encoding="utf-8-sig")
pd.concat([cmp_tr.assign(split="train"), cmp_val.assign(split="validation")]).reset_index().to_csv(OUT_DIR / "validation_and_train_comparison.csv", index=False, encoding="utf-8-sig")

In [ ]:
# F1 token cho company/address (đề yêu cầu báo F1 trường khi có quy tắc rõ)
f1 = []
for sp, d, p, b in [("train", tr, pt, bt), ("validation", va, pv, bv)]:
    for f in ["company", "address"]:
        fb = np.mean([token_f1(x, g) for x, g in zip(b[f + "_pred"], d[f])]); fi = np.mean([token_f1(x, g) for x, g in zip(p[f], d[f])])
        f1.append(dict(split=sp, field=f, baseline_F1=round(fb, 4), cai_tien_F1=round(fi, 4), chenh_lech=round(fi - fb, 4)))
f1 = pd.DataFrame(f1); display(f1); f1.to_csv(OUT_DIR / "f1_company_address.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Các ca baseline ĐÚNG nhưng cải tiến SAI (bị làm hỏng) – xem để chỉnh luật hoặc gửi cho người hỗ trợ
def show_regressions(field, d=va, p=pv, b=bv, n=3):
    shown = 0
    for i in range(len(d)):
        if b.at[i, field + "_em"] == 1 and exact(p.at[i, field], d.at[i, field], field) == 0:
            print("=" * 60); print("KEY:", d.at[i, "key"], "|", field)
            print("BASELINE:", b.at[i, field + "_pred"]); print("CẢI TIẾN:", p.at[i, field]); print("ĐÁP ÁN  :", d.at[i, field])
            print("--- OCR (700 ký tự đầu) ---"); print(d.at[i, "ocr_text"][:700]); shown += 1
            if shown >= n: break
    if not shown: print(field, ": không có ca nào bị làm hỏng.")
for f in FIELDS: print("#" * 20, f, "- bị làm hỏng:", int(((bv[f + "_em"] == 1) & (np.array([exact(a, g, f) for a, g in zip(pv[f], va[f])]) == 0)).sum()))
# show_regressions("address"); show_regressions("total")   # bỏ dấu # để xem ví dụ

In [ ]:
# Sau cải tiến còn bao nhiêu lỗi do luật / do OCR (Validation)
iv = va[["key"]].copy()
for f in FIELDS: iv[f + "_em"] = [exact(a, g, f) for a, g in zip(pv[f], va[f])]; iv[f + "_gold"] = va[f].values
print("Cải tiến – Validation"); display(attribution(iv, va.ocr_text))

## 10. Kết luận theo số liệu

In [ ]:
g = cmp_val.loc["MACRO"]
if g.chenh_lech > 0 and g.sua_dung > g.lam_hong:
    verdict = f"Cải tiến tốt hơn baseline trên Validation: Macro EM {g.baseline:.3f} -> {g.cai_tien:.3f} ({g.chenh_lech:+.3f}); sửa đúng {int(g.sua_dung)}, làm hỏng {int(g.lam_hong)} lượt trường/ảnh."
elif g.chenh_lech == 0 and g.sua_dung == g.lam_hong:
    verdict = "Không có khác biệt trên Validation: KHÔNG tuyên bố cải tiến tốt hơn baseline."
else:
    verdict = f"Cải tiến KHÔNG tốt hơn baseline trên Validation (chênh lệch {g.chenh_lech:+.3f}; sửa {int(g.sua_dung)}, hỏng {int(g.lam_hong)})."
print(verdict)
for f in FIELDS: print(f"  {f:8s}: {cmp_val.loc[f,'baseline']:.3f} -> {cmp_val.loc[f,'cai_tien']:.3f} ({cmp_val.loc[f,'chenh_lech']:+.3f})")
print("Lưu ý: Validation chỉ %d ảnh nên chênh lệch vài ảnh chưa kết luận mạnh; cần đối chiếu thêm với Train (mục 9)." % len(va))

## 11. Xuất file và bàn giao cho người 5

In [ ]:
rows = []
for d, p in [(tr, pt), (va, pv)]:
    for i in d.index:
        r = {"key": d.at[i, "key"], "split": d.at[i, "split"], "ocr_text_raw": d.at[i, "ocr_text"]}
        for f in FIELDS:
            r[f + "_pred_raw"] = p.at[i, f]; r[f + "_pred_norm"] = NORM[f](p.at[i, f])
            r[f + "_gold"] = d.at[i, f]; r[f + "_gold_norm"] = NORM[f](d.at[i, f]); r[f + "_em"] = exact(p.at[i, f], d.at[i, f], f)
            if f in ("company", "address"): r[f + "_f1"] = token_f1(p.at[i, f], d.at[i, f])
        rows.append(r)
improved = pd.DataFrame(rows)
for sp, name in [("train", "improved_train.csv"), ("validation", "improved_validation.csv")]:
    improved[improved.split == sp].to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")

DESC = {"clean_date":"Sửa nhầm ký tự O/0, l/1, S/5 trong chuỗi ngày","date_v2":"Kiểm tra ngày hợp lệ, nhiều định dạng, ưu tiên dòng DATE, bỏ ngày hết hạn",
 "clean_amount":"Sửa nhầm ký tự trong số tiền","total_kw":"Thêm từ khóa total, không loại nhầm TOTAL INCL. GST",
 "total_nospace":"So khớp từ khóa bỏ khoảng trắng (NetTotal)","total_gstcut":"Bỏ phần GST/TAX SUMMARY khi tìm total",
 "total_pos":"Không có từ khóa → số lớn nhất ở phần dưới hóa đơn","clean_text":"Làm sạch ký tự rác/dòng ký hiệu (đo bằng macro)",
 "company_v2":"Chấm điểm dòng tên công ty (hậu tố, chữ HOA, loại nhiễu), ghép SDN BHD bị tách dòng",
 "addr_v2":"Dấu hiệu địa chỉ rộng hơn, cắt TEL/FAX cuối dòng","addr_merge":"Lấy khối dòng địa chỉ liên tiếp (cho phép hỏng 1 dòng)"}
A = abl.pivot(index="config", columns="split")
tb = []
for flag in DEFAULT_CFG:
    f = FLAG_FIELD[flag]; k, s = f"+ {flag}", BASE
    tb.append(dict(co=flag, truong=f, mo_ta=DESC[flag],
                   EM_train_baseline=A.loc[s, (f, "Train")], EM_train_chi_bat_co=A.loc[k, (f, "Train")],
                   EM_val_baseline=A.loc[s, (f, "Validation")], EM_val_chi_bat_co=A.loc[k, (f, "Validation")],
                   duoc_chon="có" if chosen[flag] else "không"))
tb = pd.DataFrame(tb); tb.to_csv(OUT_DIR / "improvement_explanations.csv", index=False, encoding="utf-8-sig"); display(tb)

README = f'''# Bàn giao người 4 -> người 5
Cấu hình đóng băng (chọn trên {SELECT_ON}): `outputs/improved/frozen_config.json`. Không chỉnh sau khi xem Validation/Test.
Kết quả: `improved_train.csv`, `improved_validation.csv`, `validation_and_train_comparison.csv`, `f1_company_address.csv`, `improvement_explanations.csv`, `ablation_tung_co.csv`.
## Cách dùng (người 5)
```python
from src.improved import ocr_frozen, extract_frozen, norm_date, norm_total, norm_text, exact
text = ocr_frozen(pil_image)             # (nếu cần OCR từ ảnh) dùng đúng bước tiền xử lý đã đóng băng
pred = extract_frozen(text)              # dict company/date/address/total (giá trị GỐC)
ok = exact(pred["total"], gold_total, "total")   # so khớp sau chuẩn hóa
```
Cấu hình đọc từ `outputs/improved/frozen_config.json`. Chạy trên Test đúng một lần, không chỉnh luật sau đó.
Nguồn OCR: {OCR_SOURCE}. Ghi chú: nếu mục 5 còn tắt (RUN_PREPROC=False) thì chưa thử tiền xử lý ảnh.

## Kết luận (Validation, n={len(va)})
{verdict}
'''
(OUT_DIR / "HUONG_DAN_NGUOI_5.md").write_text(README, encoding="utf-8"); print(README)

## 12. Xuất mã nguồn cho người 5 (`src/improved.py`)
Cell này ghi toàn bộ luật cải tiến + hàm chuẩn hóa + hàm `extract_frozen()` thành file `src/improved.py`, để người 5 import mà không phải chạy lại notebook.

In [ ]:
HANDOFF_SRC = '"""improved.py - phuong phap cai tien (Nguoi 4). Tu dong sinh tu 04_cai_tien.ipynb."""\nimport re\nfrom datetime import date as _date\n\nFIELDS = ["company", "date", "address", "total"]\n\nDEFAULT_CFG = dict(\n    clean_text=False,     # lam sach dong OCR: bo ky tu rac dau dong, \\x0c, dong chi co ky hieu\n    clean_date=False,     # sua nham O/0, l/1, S/5... trong chuoi ngay (truoc khi tim ngay)\n    date_v2=False,        # kiem tra ngay hop le, nhieu dinh dang, uu tien dong DATE, bo ngay het han\n    clean_amount=False,   # sua nham ky tu so trong so tien (truoc khi tim total)\n    total_kw=False,       # tu khoa total mo rong + khong loai nham \'TOTAL INCL GST\'\n    total_nospace=False,  # so khop tu khoa sau khi bo khoang trang (NetTotal, GRANDTOTAL)\n    total_gstcut=False,   # bo phan GST/TAX SUMMARY khi tim total\n    total_pos=False,      # khong co tu khoa -> so lon nhat o phan duoi hoa don\n    company_v2=False,     # cham diem cac dong dau de chon ten cong ty (+ ghep SDN BHD tach dong)\n    addr_v2=False,        # dia chi: dau hieu dia chi rong hon, cat phan TEL/FAX cuoi dong\n    addr_merge=False,     # dia chi: lay KHOI dong lien tiep (cho phep hong 1 dong) thay vi gom roi rac\n)\n\n# ================================================================ BASELINE (sao chep tu src/baseline.py cua nguoi 3)\nMONTHS = r"(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*"\nDATE_PATTERNS = [\n    r"\\b\\d{1,2}\\s*[/\\-.]\\s*\\d{1,2}\\s*[/\\-.]\\s*\\d{2,4}\\b",\n    r"\\b\\d{4}\\s*[/\\-.]\\s*\\d{1,2}\\s*[/\\-.]\\s*\\d{1,2}\\b",\n    r"\\b\\d{1,2}[\\s\\-]*" + MONTHS + r"[\\s\\-,]*\\d{2,4}\\b",\n]\nDATE_RE = re.compile("|".join(f"(?:{p})" for p in DATE_PATTERNS), re.IGNORECASE)\nAMOUNT_RE = re.compile(r"\\d[\\d,]*[.,]\\s?\\d{2}\\b")\nTOTAL_KEYS = ["GRAND TOTAL", "TOTAL ROUNDED", "ROUNDED TOTAL", "NET TOTAL",\n              "TOTAL AMOUNT", "AMOUNT DUE", "AMOUNT PAYABLE", "TOTAL"]\nTOTAL_EXCLUDE = re.compile(r"SUB|EXCL|QTY|QUANTITY|SAVING|DISCOUNT|POINT|TOTAL\\s*GST|TAX", re.IGNORECASE)\nADDR_KW = re.compile(\n    r"\\b(JALAN|JLN|TAMAN|TMN|LOT|NO|BANDAR|KAWASAN|SELANGOR|JOHOR|KUALA LUMPUR|KL|"\n    r"PETALING|SHAH ALAM|KLANG|CHERAS|PAHANG|PENANG|WILAYAH)\\b", re.IGNORECASE)\nPOSTCODE = re.compile(r"\\b\\d{5}\\b")\nADDR_STOP = re.compile(r"\\b(TEL|FAX|GST|INVOICE|RECEIPT|DATE|CASH|BILL|TAX|DOC|CASHIER)\\b", re.IGNORECASE)\n\n\ndef _get_lines(ocr_text):\n    if not isinstance(ocr_text, str):\n        return []\n    return [ln.strip() for ln in ocr_text.split("\\n") if ln.strip()]\n\n\ndef _b_company(lines):\n    for i, ln in enumerate(lines[:5]):\n        if sum(c.isalpha() for c in ln) >= 3:\n            return ln, i\n    return None, -1\n\n\ndef _b_date(lines):\n    for ln in lines:\n        if "DATE" in ln.upper():\n            m = DATE_RE.search(ln)\n            if m:\n                return m.group(0).strip()\n    m = DATE_RE.search("\\n".join(lines))\n    return m.group(0).strip() if m else None\n\n\ndef _b_address(lines, company_idx):\n    if company_idx < 0:\n        return None\n    parts = []\n    for ln in lines[company_idx + 1: company_idx + 9]:\n        if ADDR_STOP.search(ln):\n            break\n        if ADDR_KW.search(ln) or POSTCODE.search(ln):\n            parts.append(ln)\n    return " ".join(parts) if parts else None\n\n\n# ================================================================ lam sach\n_CONF = str.maketrans("OoIl|SsBZ", "001115582")\n_TOKEN = re.compile(r"[0-9A-Za-z|./,\\-]{4,}")\n_ALLOWED = set("0123456789OoIl|SsBZ./,-")\n\n\ndef _digit_fix(line):\n    """Trong token co >= 2 chu so va chi gom ky tu de nham, doi O->0, l->1, S->5..."""\n    def fn(m):\n        t = m.group(0)\n        if sum(c.isdigit() for c in t) >= 2 and all(c in _ALLOWED for c in t):\n            return t.translate(_CONF)\n        return t\n    return _TOKEN.sub(fn, line)\n\n\ndef _clean_lines(lines):\n    out = []\n    for l in lines:\n        l = l.replace("\\x0c", " ")\n        l = re.sub(r"^[^A-Za-z0-9(]+", "", l)\n        l = re.sub(r"\\s+", " ", l).strip()\n        if sum(c.isalnum() for c in l) >= 2:\n            out.append(l)\n    return out\n\n\n# ================================================================ date\n_MONTHS = {m: i + 1 for i, m in enumerate("JAN FEB MAR APR MAY JUN JUL AUG SEP OCT NOV DEC".split())}\n_MON = "|".join(_MONTHS)\n_P_NUM = re.compile(r"(?<!\\d)(\\d{1,2})\\s*[/\\-.]\\s*(\\d{1,2})\\s*[/\\-.]\\s*(\\d{4}|\\d{2})(?!\\d)")\n_P_ISO = re.compile(r"(?<!\\d)(\\d{4})\\s*[/\\-.]\\s*(\\d{1,2})\\s*[/\\-.]\\s*(\\d{1,2})(?!\\d)")\n_P_TXT = re.compile(r"(?<!\\d)(\\d{1,2})\\s*[\\- ]?\\s*(" + _MON + r")[A-Z]*\\.?[\\s,\\-]*(\\d{4}|\\d{2})(?!\\d)")\n_P_TXT2 = re.compile(r"(" + _MON + r")[A-Z]*\\.?\\s*(\\d{1,2})(?:ST|ND|RD|TH)?[\\s,\\-]+(\\d{4})(?!\\d)")\n\n\ndef _mk(y, m, d):\n    y, m, d = int(y), int(m), int(d)\n    if y < 100:\n        y += 2000\n    if not 2000 <= y <= 2030:\n        return None\n    try:\n        return _date(y, m, d).isoformat()\n    except ValueError:\n        return None\n\n\ndef _find_dates_line(u):\n    """-> list[(vi_tri, iso, chuoi_goc)] chi gom ngay HOP LE."""\n    out = []\n    for m in _P_ISO.finditer(u):\n        iso = _mk(*m.groups())\n        if iso:\n            out.append((m.start(), iso, m.group(0)))\n    for m in _P_NUM.finditer(u):\n        a, b, c = m.groups()\n        a_, b_ = int(a), int(b)\n        if b_ > 12 and a_ <= 12:\n            a_, b_ = b_, a_\n        iso = _mk(c, b_, a_)\n        if iso:\n            out.append((m.start(), iso, m.group(0)))\n    for m in _P_TXT.finditer(u):\n        d, mon, y = m.groups()\n        iso = _mk(y, _MONTHS[mon], d)\n        if iso:\n            out.append((m.start(), iso, m.group(0)))\n    for m in _P_TXT2.finditer(u):\n        mon, d, y = m.groups()\n        iso = _mk(y, _MONTHS[mon], d)\n        if iso:\n            out.append((m.start(), iso, m.group(0)))\n    return sorted(out)\n\n\ndef _v2_date(lines):\n    best = None\n    for i, l in enumerate(lines):\n        u = l.upper()\n        for pos, iso, raw in _find_dates_line(u):\n            score = 2 if "DATE" in u else 0\n            if re.search(r"EXP|DUE|VALID|BEST BEFORE", u):\n                score -= 2\n            key = (score, -i, -pos)\n            if best is None or key > best[0]:\n                best = (key, raw)\n    return best[1].strip() if best else None\n\n\ndef extract_date(lines, cfg):\n    if cfg["clean_date"]:\n        lines = [_digit_fix(l) for l in lines]\n    if cfg["date_v2"]:\n        d = _v2_date(lines)\n        if d:\n            return d\n    return _b_date(lines)\n\n\n# ================================================================ total\n_TOTAL_KEYS_V2 = ["GRAND TOTAL", "TOTAL ROUNDED", "ROUNDED TOTAL", "NET TOTAL", "NETT TOTAL", "TOTAL AMOUNT",\n                  "AMOUNT DUE", "TOTAL DUE", "AMOUNT PAYABLE", "TOTAL PAYABLE", "TOTAL INCL", "TOTAL SALES", "TOTAL"]\n_TOTAL_EXCLUDE_V2 = re.compile(r"SUB|EXCL|QTY|QUANTITY|SAVING|DISCOUNT|POINT|CHANGE|TENDER|TOTAL\\s*(?:GST|TAX)|"\n                               r"(?:GST|TAX)\\s*TOTAL", re.IGNORECASE)\n_GST_HEAD = re.compile(r"(?:GST|TAX)\\s*(?:SUMMARY|ANALYSIS)", re.IGNORECASE)\n\n\ndef _amount_value(a):\n    s = a.replace(" ", "")\n    if re.search(r",\\d{2}$", s):\n        s = s[:-3] + "." + s[-2:]\n    return float(s.replace(",", ""))\n\n\ndef extract_total(lines, cfg):\n    if cfg["clean_amount"]:\n        lines = [_digit_fix(l) for l in lines]\n    if cfg["total_gstcut"]:\n        for i, l in enumerate(lines):\n            if _GST_HEAD.search(l):\n                lines = lines[:i]\n                break\n    keys = _TOTAL_KEYS_V2 if cfg["total_kw"] else TOTAL_KEYS\n    excl = _TOTAL_EXCLUDE_V2 if cfg["total_kw"] else TOTAL_EXCLUDE\n    ns = cfg["total_nospace"]\n    for key in keys:\n        k = key.replace(" ", "") if ns else key\n        for i, ln in enumerate(lines):\n            up = ln.upper().replace(" ", "") if ns else ln.upper()\n            if k in up and not excl.search(ln.upper()):\n                amounts = AMOUNT_RE.findall(ln)\n                if not amounts and i + 1 < len(lines):\n                    amounts = AMOUNT_RE.findall(lines[i + 1])\n                if amounts:\n                    return amounts[-1].replace(" ", "")\n    if cfg["total_pos"]:\n        n = len(lines)\n        cands = []\n        for i, ln in enumerate(lines):\n            if i >= n * 0.4 and not excl.search(ln.upper()):\n                for a in AMOUNT_RE.findall(ln):\n                    try:\n                        cands.append((_amount_value(a), a))\n                    except ValueError:\n                        pass\n        if cands:\n            return max(cands)[1].replace(" ", "")\n    return None\n\n\n# ================================================================ company\n_LEGAL = re.compile(r"\\b(SDN\\.?\\s*BHD|BHD|S/?B|ENTERPRISE|TRADING|RESTAURANT|RESTORAN|STORE|MARKET|MART|CAFE|"\n                    r"BAKERY|PHARMACY|LTD|INC|CO\\.?|PLT|CORPORATION|HOLDINGS|SUPERMARKET|HARDWARE|BOOK|"\n                    r"GIFT|HOME|MACHINERY|SHOP|MEDICAL|CLINIC|KEDAI)\\b")\n_NOISE = re.compile(r"\\b(TEL|FAX|PHONE|INVOICE|RECEIPT|TAX|DATE|GST|REG|CASHIER|DOC|BILL|WWW|EMAIL|HP)\\b|\\bNO\\s*[:.]")\n_ADDR_CUE = re.compile(r"\\b(JALAN|JLN|LORONG|LRG|TAMAN|TMN|BANDAR|KAWASAN|LOT|FLOOR|LEVEL|BLOCK|BLK|MENARA|WISMA|"\n                       r"SEKSYEN|PUSAT|KOMPLEKS|MALAYSIA|KUALA LUMPUR|SELANGOR|JOHOR|PENANG|PERAK|KEDAH|SABAH|"\n                       r"SARAWAK|MELAKA|PAHANG|PETALING|SHAH ALAM|KLANG|CHERAS|WILAYAH|KL|NO)\\b", re.IGNORECASE)\n_ADDR_STOP2 = re.compile(r"\\b(TEL|FAX|PHONE|GST|INVOICE|RECEIPT|DATE|DOC|CASHIER|EMAIL|WWW|HP|BILL|SLIP|CASH|TAX)\\b",\n                         re.IGNORECASE)\n\n\ndef _v2_company(lines):\n    best = None\n    for rank, i in enumerate(range(min(len(lines), 8))):\n        l = lines[i].strip()\n        u = l.upper()\n        if len(l) < 3 or sum(c.isalpha() for c in l) < 3:\n            continue\n        s = -0.3 * rank\n        if _LEGAL.search(u):\n            s += 3\n        if _NOISE.search(u):\n            s -= 4\n        if _ADDR_CUE.search(u) or re.search(r"\\b\\d{5}\\b", u):\n            s -= 3\n        if sum(c.isdigit() for c in l) / len(l) > 0.3:\n            s -= 2\n        letters = [c for c in l if c.isalpha()]\n        up = sum(c.isupper() for c in letters) / len(letters)\n        s += 1 if up >= 0.8 else (-1.5 if up == 0 else 0)\n        if best is None or s > best[0]:\n            best = (s, i)\n    if best is None:\n        return None, -1\n    i = best[1]\n    name, end = lines[i].strip(), i\n    if (i + 1 < len(lines) and not _LEGAL.search(name.upper())\n            and re.match(r"^\\s*(SDN\\.?\\s*BHD|BHD|\\(?S/?B\\)?)\\b", lines[i + 1].upper())):\n        name, end = name + " " + lines[i + 1].strip(), i + 1\n    return name, end\n\n\n# ================================================================ address\ndef _cut_tail(l):\n    return re.split(r"\\b(?:TEL|PHONE|FAX|HP)\\b", l, maxsplit=1, flags=re.I)[0].strip(" ,:-")\n\n\ndef extract_address(lines, cidx, cfg):\n    if cidx < 0:\n        return None\n    if not (cfg["addr_v2"] or cfg["addr_merge"]):\n        return _b_address(lines, cidx)           # = baseline, nhung tinh tu dong company da chon\n    v2 = cfg["addr_v2"]\n    kw = _ADDR_CUE if v2 else ADDR_KW\n    stop = _ADDR_STOP2 if v2 else ADDR_STOP\n    window = lines[cidx + 1: cidx + 9]\n    fix = _cut_tail if v2 else (lambda x: x)\n    if not cfg["addr_merge"]:                    # chi addr_v2: gom nhu baseline nhung dau hieu rong hon + cat TEL\n        parts = []\n        for ln in window:\n            if stop.search(ln):\n                break\n            if kw.search(ln) or POSTCODE.search(ln):\n                parts.append(fix(ln))\n        return " ".join(p for p in parts if p) or None\n    # addr_merge: lay KHOI dong lien tiep co dau hieu dia chi, cho phep hong toi da 1 dong giua khoi\n    parts, started, gap = [], False, 0\n    for ln in window:\n        if stop.search(ln):\n            break\n        if kw.search(ln) or POSTCODE.search(ln):\n            parts.append(fix(ln))\n            started, gap = True, 0\n        elif started:\n            gap += 1\n            if gap > 1:\n                break\n            parts.append(("?", fix(ln)))\n    out = []\n    for p in parts:\n        out.append(p)\n    # bo dong \'hong\' o cuoi khoi, giu dong \'hong\' nam giua hai dong dia chi\n    while out and isinstance(out[-1], tuple):\n        out.pop()\n    return " ".join(p[1] if isinstance(p, tuple) else p for p in out if (p[1] if isinstance(p, tuple) else p)) or None\n\n\n# ================================================================ API\ndef extract(text, cfg=None):\n    cfg = {**DEFAULT_CFG, **(cfg or {})}\n    lines = _get_lines(text)\n    if cfg["clean_text"]:\n        lines = _clean_lines(lines)\n    if cfg["company_v2"]:\n        company, cidx = _v2_company(lines)\n    else:\n        company, cidx = _b_company(lines)\n    return dict(company=company, date=extract_date(lines, cfg),\n                address=extract_address(lines, cidx, cfg), total=extract_total(lines, cfg))\n\n\n\n# ---- tien xu ly anh + OCR (giong 02_ocr.ipynb cua nguoi 1: Tesseract, --oem 3 --psm 6, lang eng) ----\nOCR_CONFIG = "--oem 3 --psm 6"\nREF_STEPS = ["gray", "resize2"]      # dung nhu OCR CSV cua nguoi 1: grayscale + phong 2x (INTER_CUBIC)\n\n\ndef _deskew(gray):\n    import cv2\n    import numpy as np\n    ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]\n    ys, xs = np.where(ink > 0)\n    if len(xs) < 200:\n        return gray\n    ang = cv2.minAreaRect(np.column_stack([xs, ys]).astype(np.float32))[-1]\n    ang = ang - 90 if ang > 45 else ang\n    if not 0.3 <= abs(ang) <= 15:\n        return gray\n    h, w = gray.shape[:2]\n\n    def rot(a):\n        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)\n        return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)\n\n    def sharp(img):   # dong chu thang -> tong theo hang dao dong manh (tranh nham dau goc giua cac ban OpenCV)\n        b = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]\n        return float(np.var(b.sum(axis=1)))\n\n    r1, r2 = rot(ang), rot(-ang)\n    return max([(sharp(gray), gray), (sharp(r1), r1), (sharp(r2), r2)], key=lambda c: c[0])[1]\n\n\ndef preprocess_image(rgb, steps):\n    """rgb: ndarray HxWx3 (RGB); steps: danh sach buoc ap dung theo thu tu."""\n    import cv2\n    out = rgb\n    for st in steps:\n        if st == "gray":\n            out = cv2.cvtColor(out, cv2.COLOR_RGB2GRAY) if out.ndim == 3 else out\n        elif st in ("resize2", "resize3"):\n            f = 2 if st == "resize2" else 3\n            out = cv2.resize(out, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)\n        elif st == "median":\n            out = cv2.medianBlur(out, 3)\n        elif st == "clahe":\n            out = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(out)\n        elif st == "otsu":\n            out = cv2.threshold(cv2.GaussianBlur(out, (3, 3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]\n        elif st == "adaptive":\n            out = cv2.adaptiveThreshold(cv2.GaussianBlur(out, (3, 3), 0), 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,\n                                        cv2.THRESH_BINARY, 31, 15)\n        elif st == "deskew":\n            out = _deskew(out)\n        else:\n            raise ValueError("buoc tien xu ly khong biet: " + st)\n    return out\n\n\ndef ocr_image(pil_image, steps=None):\n    import numpy as np\n    import pytesseract\n    steps = REF_STEPS if steps is None else steps\n    img = preprocess_image(np.array(pil_image.convert("RGB")), steps)\n    return pytesseract.image_to_string(img, lang="eng", config=OCR_CONFIG)\n\n\n# ---------------- ham chuan hoa + tai cau hinh dong bang (danh cho nguoi 5) ----------------\nfrom datetime import datetime as _dt\nimport json as _json\nfrom pathlib import Path as _Path\n\n_DATE_FORMATS = ["%d/%m/%Y","%d-%m-%Y","%d.%m.%Y","%d/%m/%y","%d-%m-%y","%d.%m.%y","%Y-%m-%d","%Y/%m/%d",\n                 "%d %b %Y","%d%b%Y","%d-%b-%Y","%d %b %y","%d %B %Y"]\n\ndef is_missing(x):\n    return x is None or (isinstance(x, float) and x != x) or str(x).strip() == ""\n\ndef norm_text(s):\n    if is_missing(s): return None\n    s = re.sub(r"[^\\w\\s]", " ", str(s).lower()); return re.sub(r"\\s+", " ", s).strip()\n\ndef norm_total(s):\n    if is_missing(s): return None\n    nums = re.findall(r"\\d[\\d,.]*", str(s).replace(" ", ""))\n    if not nums: return None\n    tok = nums[-1]\n    if re.search(r",\\d{2}$", tok): tok = tok[:-3] + "." + tok[-2:]\n    tok = tok.replace(",", "")\n    try: return f"{float(tok):.2f}"\n    except ValueError: return None\n\ndef norm_date(s):\n    if is_missing(s): return None\n    core = re.sub(r"\\s*([/\\-.])\\s*", r"\\1", str(s).strip())\n    for cand in (core, core.replace(" ", "")):\n        for f in _DATE_FORMATS:\n            try: return _dt.strptime(cand, f).strftime("%Y-%m-%d")\n            except ValueError: pass\n    return norm_text(core)\n\nNORM = {"company": norm_text, "address": norm_text, "date": norm_date, "total": norm_total}\n\ndef exact(pred, gold, field):\n    if is_missing(pred) or is_missing(gold): return 0\n    return int(NORM[field](pred) == NORM[field](gold))\n\ndef load_frozen_cfg(path="outputs/improved/frozen_config.json"):\n    return _json.loads(_Path(path).read_text(encoding="utf-8"))["cfg"]\n\ndef load_frozen_ocr_steps(path="outputs/improved/frozen_config.json"):\n    return _json.loads(_Path(path).read_text(encoding="utf-8")).get("ocr", {}).get("steps", REF_STEPS)\n\ndef ocr_frozen(pil_image, path="outputs/improved/frozen_config.json"):\n    """Anh (PIL) -> van ban OCR bang dung buoc tien xu ly da dong bang."""\n    return ocr_image(pil_image, load_frozen_ocr_steps(path))\n\ndef extract_frozen(ocr_text, path="outputs/improved/frozen_config.json"):\n    """Phuong phap cai tien voi cau hinh DONG BANG. Tra ve dict gia tri GOC (chua chuan hoa)."""\n    return extract(ocr_text, load_frozen_cfg(path))\n'
(PROJECT_DIR / "src").mkdir(exist_ok=True)
(PROJECT_DIR / "src" / "improved.py").write_text(HANDOFF_SRC, encoding="utf-8")
print("Đã ghi", PROJECT_DIR / "src" / "improved.py")